# EXP-24: Specialist Ensemble — Targeted Rare Class Boost

**Competition:** SPR 2026 — Mammography Report Classification  
**Insight:** Todos os experimentos que tentaram "mais modelos" ou "mais features" pioraram.  
O problema real são as **classes raras** (organizer confirmou).

**Approach:**
- Motor principal: V12 ensemble (3 SVCs + 1 LGB) — comprovado (0.804)
- Especialistas: Binary SVCs one-vs-rest para classes raras (0, 1, 3, 5, 6)
- Blending inteligente: ajusta predições do ensemble principal com confiança dos especialistas
- GroupKFold 5-fold CV para validação local
- Thresholds otimizados por OOF para classes 3-6
**Runtime:** CPU only

In [ ]:
import os, re, hashlib, warnings, time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb
warnings.filterwarnings('ignore')

TRAIN_PATH = '/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv'
TEST_PATH  = '/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

N_FOLDS = 5
N_CLASSES = 7
RARE_CLASSES = [0, 1, 3, 5, 6]

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Class distribution:\n{train["target"].value_counts().sort_index()}')
print(f'\nRare classes: {RARE_CLASSES}')

In [ ]:
def clean_achados(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{2,}", "\n", s)
    match = re.search(r'achados:(.*?)(análise comparativa:|$)', s, flags=re.DOTALL)
    if match: s = match.group(1).strip()
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def clean_full(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def stable_hash(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def extract_dense_features(df):
    features = pd.DataFrame(index=df.index)
    text_col = df['report'].fillna('').astype(str).str.lower()
    features['report_length']   = text_col.apply(len)
    features['has_measurement'] = text_col.str.contains(r'\b(?:cm|mm|medindo)\b', regex=True).astype(int)
    features['has_spiculation'] = text_col.str.contains(r'espiculad', regex=True).astype(int)
    features['has_distortion']  = text_col.str.contains(r'distorção arquitetural', regex=True).astype(int)
    features['has_biopsy']      = text_col.str.contains(r'biopsy|biópsia|resultado de cine|carcinoma', regex=True).astype(int)
    return csr_matrix(features.values)

train["achados"] = train["report"].apply(clean_achados)
train["full"]    = train["report"].apply(clean_full)
test["achados"]  = test["report"].apply(clean_achados)
test["full"]     = test["report"].apply(clean_full)
train["group"]   = train["report"].apply(stable_hash)

X_train_dense = extract_dense_features(train)
X_test_dense  = extract_dense_features(test)
y = train["target"].astype(int).values
groups = train["group"].values

print("Preprocessing done.")

In [ ]:
print("Building TF-IDF features...")

tfidf_A = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
X_train_A = tfidf_A.fit_transform(train["achados"].values)
X_test_A  = tfidf_A.transform(test["achados"].values)

tfidf_F = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
X_train_F = tfidf_F.fit_transform(train["full"].values)
X_test_F  = tfidf_F.transform(test["full"].values)

tfidf_F2 = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=100000))
])
X_train_F2 = tfidf_F2.fit_transform(train["full"].values)
X_test_F2  = tfidf_F2.transform(test["full"].values)

X_train_lgb = hstack([X_train_A, X_train_dense]).tocsr()
X_test_lgb  = hstack([X_test_A,  X_test_dense]).tocsr()

# Combined features for specialists (full + achados + dense)
X_train_combined = hstack([X_train_F, X_train_A, X_train_dense]).tocsr()
X_test_combined  = hstack([X_test_F, X_test_A, X_test_dense]).tocsr()

print(f"SVC-A: {X_train_A.shape[1]}, SVC-F: {X_train_F.shape[1]}, SVC-F2: {X_train_F2.shape[1]}")
print(f"LGB: {X_train_lgb.shape[1]}, Combined (for specialists): {X_train_combined.shape[1]}")

In [ ]:
# =============================================================================
# Part 1: Main Engine — V12-style multiclass ensemble (proven)
# 3 SVCs + 1 LGB with GroupKFold CV
# =============================================================================

gkf = GroupKFold(n_splits=N_FOLDS)
folds = list(gkf.split(X_train_A, y, groups))

main_oof = {}
main_test = {}

t0 = time.time()

# SVC on achados
for name, X_tr, X_te in [('svc_A', X_train_A, X_test_A),
                           ('svc_F', X_train_F, X_test_F),
                           ('svc_F2', X_train_F2, X_test_F2)]:
    print(f"\n--- Main: {name} ---")
    oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
    te_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
    for fi, (tr_idx, va_idx) in enumerate(folds):
        m = CalibratedClassifierCV(
            LinearSVC(class_weight="balanced", random_state=42, max_iter=10000),
            cv=3, method='sigmoid'
        )
        m.fit(X_tr[tr_idx], y[tr_idx])
        oof[va_idx] = m.predict_proba(X_tr[va_idx])
        te_acc += m.predict_proba(X_te) / N_FOLDS
    print(f"  OOF F1: {f1_score(y, np.argmax(oof, axis=1), average='macro'):.4f}")
    main_oof[name] = oof
    main_test[name] = te_acc

# LGB on achados + dense
print(f"\n--- Main: lgb ---")
oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
te_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
for fi, (tr_idx, va_idx) in enumerate(folds):
    m = lgb.LGBMClassifier(class_weight='balanced', n_estimators=300, learning_rate=0.05,
                           max_depth=6, random_state=42, n_jobs=-1, verbose=-1)
    m.fit(X_train_lgb[tr_idx], y[tr_idx])
    oof[va_idx] = m.predict_proba(X_train_lgb[va_idx])
    te_acc += m.predict_proba(X_test_lgb) / N_FOLDS
print(f"  OOF F1: {f1_score(y, np.argmax(oof, axis=1), average='macro'):.4f}")
main_oof['lgb'] = oof
main_test['lgb'] = te_acc

# V12 blend
oof_svc_ens = 0.25 * main_oof['svc_A'] + 0.40 * main_oof['svc_F'] + 0.35 * main_oof['svc_F2']
oof_main = 0.70 * oof_svc_ens + 0.30 * main_oof['lgb']

te_svc_ens = 0.25 * main_test['svc_A'] + 0.40 * main_test['svc_F'] + 0.35 * main_test['svc_F2']
te_main = 0.70 * te_svc_ens + 0.30 * main_test['lgb']

main_f1 = f1_score(y, np.argmax(oof_main, axis=1), average='macro')
print(f"\nMain engine OOF F1: {main_f1:.4f}")
print(f"Trained main engine in {time.time()-t0:.0f}s")

In [ ]:
# =============================================================================
# Part 2: Specialist Binary SVCs — one-vs-rest for each rare class
# Uses combined features (full + achados + dense) for maximum signal
# =============================================================================

print("Training specialist binary classifiers for rare classes...")

specialist_oof = {}   # class -> (n_train,) probability of being that class
specialist_test = {}  # class -> (n_test,) probability

for cls in RARE_CLASSES:
    y_binary = (y == cls).astype(int)
    n_pos = y_binary.sum()
    n_neg = len(y_binary) - n_pos
    print(f"\n--- Specialist class {cls} (pos={n_pos}, neg={n_neg}, ratio={n_pos/len(y_binary)*100:.2f}%) ---")
    
    oof_prob = np.zeros(len(train), dtype=np.float64)
    te_prob = np.zeros(len(test), dtype=np.float64)
    
    for fi, (tr_idx, va_idx) in enumerate(folds):
        m = CalibratedClassifierCV(
            LinearSVC(class_weight='balanced', random_state=42, max_iter=10000),
            cv=3, method='sigmoid'
        )
        m.fit(X_train_combined[tr_idx], y_binary[tr_idx])
        oof_prob[va_idx] = m.predict_proba(X_train_combined[va_idx])[:, 1]
        te_prob += m.predict_proba(X_test_combined)[:, 1] / N_FOLDS
    
    # Binary specialist F1
    best_thr = 0.5
    best_bin_f1 = 0
    for thr in np.arange(0.1, 0.9, 0.01):
        bin_preds = (oof_prob > thr).astype(int)
        bin_f1 = f1_score(y_binary, bin_preds, average='binary')
        if bin_f1 > best_bin_f1:
            best_bin_f1 = bin_f1
            best_thr = thr
    
    print(f"  Best binary F1: {best_bin_f1:.4f} at threshold {best_thr:.2f}")
    specialist_oof[cls] = oof_prob
    specialist_test[cls] = te_prob

print("\nAll specialists trained.")

In [ ]:
# =============================================================================
# Part 3: Blend main engine + specialists
# Strategy: use specialist confidence to BOOST rare class probabilities
# in the main ensemble, then re-apply argmax + thresholds
# =============================================================================

# Try different blending strengths for specialists
best_alpha = 0
best_blend_f1 = main_f1

for alpha in np.arange(0.0, 0.51, 0.05):
    # Boost main probabilities with specialist signal
    oof_boosted = oof_main.copy()
    for cls in RARE_CLASSES:
        oof_boosted[:, cls] += alpha * specialist_oof[cls]
    
    # Renormalize
    row_sums = oof_boosted.sum(axis=1, keepdims=True)
    oof_boosted = oof_boosted / row_sums
    
    preds = np.argmax(oof_boosted, axis=1)
    score = f1_score(y, preds, average='macro')
    
    if score > best_blend_f1:
        best_blend_f1 = score
        best_alpha = alpha

print(f"Best specialist alpha: {best_alpha:.2f} (F1: {best_blend_f1:.4f})")
print(f"Improvement over main engine: {best_blend_f1 - main_f1:+.4f}")

# Apply best alpha
oof_blend = oof_main.copy()
te_blend = te_main.copy()
for cls in RARE_CLASSES:
    oof_blend[:, cls] += best_alpha * specialist_oof[cls]
    te_blend[:, cls] += best_alpha * specialist_test[cls]

# Renormalize
oof_blend = oof_blend / oof_blend.sum(axis=1, keepdims=True)
te_blend = te_blend / te_blend.sum(axis=1, keepdims=True)

baseline_preds = np.argmax(oof_blend, axis=1)
baseline_f1 = f1_score(y, baseline_preds, average='macro')
print(f"\nBlended OOF F1 (no thresholds): {baseline_f1:.4f}")
print(classification_report(y, baseline_preds, digits=4))

In [ ]:
# =============================================================================
# Part 4: Threshold optimization (classes 3-6, proven to work)
# =============================================================================

threshold_classes = [6, 5, 4, 3]
threshold_range = np.arange(0.05, 0.55, 0.01)

def apply_thresholds(proba, thresholds):
    preds = np.argmax(proba, axis=1).copy()
    for cls in [6, 5, 4, 3]:
        if cls in thresholds:
            mask = proba[:, cls] > thresholds[cls]
            for higher_cls in [c for c in [6, 5, 4, 3] if c != cls]:
                if higher_cls in thresholds and threshold_classes.index(higher_cls) < threshold_classes.index(cls):
                    mask = mask & (preds != higher_cls)
            preds[mask] = cls
    return preds

best_thresholds = {}
current_f1 = baseline_f1

for cls in threshold_classes:
    best_cls_f1 = current_f1
    best_cls_thr = None
    for thr in threshold_range:
        trial = {**best_thresholds, cls: thr}
        trial_preds = apply_thresholds(oof_blend, trial)
        trial_f1 = f1_score(y, trial_preds, average='macro')
        if trial_f1 > best_cls_f1:
            best_cls_f1 = trial_f1
            best_cls_thr = thr
    if best_cls_thr is not None:
        best_thresholds[cls] = best_cls_thr
        current_f1 = best_cls_f1
        print(f"Class {cls}: threshold={best_cls_thr:.2f}, macro-F1={best_cls_f1:.4f}")
    else:
        print(f"Class {cls}: no improvement")

final_oof_preds = apply_thresholds(oof_blend, best_thresholds)
final_oof_f1 = f1_score(y, final_oof_preds, average='macro')
print(f"\nFinal OOF macro-F1: {final_oof_f1:.4f}")
print(f"Thresholds: {best_thresholds}")
print(classification_report(y, final_oof_preds, digits=4))

In [ ]:
# =============================================================================
# Submission
# =============================================================================

test_preds = apply_thresholds(te_blend, best_thresholds)

def apply_safe_rules(row):
    text = str(row['report']).lower()
    current_pred = int(row['target'])
    if re.search(r'resultado de cine grau 3|carcinoma|\bcdis\b', text):
        return 6
    if 'espiculad' in text and 'distorção' in text and current_pred < 4:
        return 5
    return current_pred

test['target'] = test_preds
test['target'] = test.apply(apply_safe_rules, axis=1)

submission = test[['ID', 'target']].copy()
submission['target'] = submission['target'].astype(int)
submission.to_csv('submission.csv', index=False)

print('Prediction distribution:')
print(submission['target'].value_counts().sort_index())
print(f'\nSubmission saved! Shape: {submission.shape}')
print(submission)